In [ ]:
from dotenv import load_dotenv
from IPython.core.display import Image
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import END, START
from langgraph.graph import MessagesState, StateGraph
from rich import print as rich_print

load_dotenv(override=True)

# 1.初始化模型
model = init_chat_model("deepseek-flash", extra_body={"thinking": {"type": "disabled"}})


# 2.创建State  MessagesState
# class LLMState(TypedDict):
#     # user_message: HumanMessage
#     # ai_message: AIMessage
#     messages: Annotated[list[BaseMessage], add]
class LLMState(MessagesState):
    pass


# 3.创建Node
def llm_node(state: LLMState):
    # 调用模型，传入str，返回 AIMessage
    response = model.invoke(state["messages"])
    return {"messages": [response]}


# 4.添加Edge，连接Node，形成Graph
builder = StateGraph(LLMState)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile(checkpointer=InMemorySaver())  # 3.4.编译成可执行图
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

# ===================4.调用graph======================
# 调用graph，传入初始的state，返回的是处理后的 state

config: RunnableConfig = {"configurable": {"thread_id": "t1"}}
result = graph.invoke(
    {
        "messages": [
            HumanMessage(content="12个苹果三个人分，每个人分几个？"),
        ]
    },
    config=config,
)
# rich_print(result)
result = graph.invoke(
    {
        "messages": [
            HumanMessage(content="那6个人呢？"),
        ]
    },
    config=config,
)
rich_print(result)